# CyberSentinel · Test de Presidio et choix des métriques

**Responsable :** Imad · **Séance 2** (12/11/2026) · Carte Kanban : *état de l'art et métriques*

Ce notebook répond aux tâches de la fiche de séance :

1. ★ **Tester Presidio en français sur 20 prompts** : ce qu'il trouve, ce qu'il rate (partie 2).
2. ★ **Chercher un jeu de données public français** : AI4Privacy OpenPII 1M, analysé dans `docs/analyse-dataset-ai4privacy.md` (rappel des chiffres en partie 5).
3. ★ **Fixer les métriques** : rappel prioritaire, taux de fausses alertes, temps de réponse par prompt (parties 3 et 4).
4. **Comparer avec les règles d'Aloïs** : ce que les règles suffisent à attraper, ce qui demande un modèle (partie 4).

La synthèse et la décision sont dans **`docs/choix-modele-metriques.md`**.

**Ce qu'on réutilise du travail de l'équipe :**

| Source | Auteur | Utilisation ici |
|---|---|---|
| `data/prompts_v1.csv` (150 prompts étiquetés) | Nolan | jeu de test « prompts réalistes » |
| `docs/guide-annotation.md` | Elias | définition de « sensible » et des pièges |
| `rules/m1.yaml` + `rules/m1_detect.py` (branche `cyber`) | Aloïs | détecteur « règles M1 » comparé à Presidio |
| `docs/interface-detect.md` (branche `cyber`) | Arno, Nolan | format de sortie de `detect()` |

**Pour relancer :** environnement avec `data/requirements.txt` + `python -m spacy download fr_core_news_lg`, puis *Run All*.
Les règles M1 sont lues dans `rules/` si la branche `cyber` a été fusionnée, sinon elles sont extraites automatiquement de `origin/cyber` (voir `data/scripts/m1_source.py`).

> Toutes les données utilisées sont **inventées** (prompts de l'équipe, dataset synthétique). Aucune donnée réelle.

## 0. Chargement

In [1]:
import sys
import time
import statistics
from collections import Counter
from pathlib import Path

import pandas as pd

RACINE = Path.cwd() if (Path.cwd() / "data" / "prompts_v1.csv").exists() else Path.cwd().parent
sys.path.insert(0, str(RACINE / "data" / "scripts"))

from presidio_detector import creer_analyzer   # configuration Presidio FR (spaCy fr_core_news_lg)
from m1_source import charger_m1               # règles M1 d'Aloïs

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

prompts = pd.read_csv(RACINE / "data" / "prompts_v1.csv", encoding="utf-8-sig")
prompts["types_donnees"] = prompts["types_donnees"].fillna("")
prompts["types"] = prompts["types_donnees"].apply(lambda s: [t for t in s.split("|") if t])

m1, origine_m1 = charger_m1()
analyzer = creer_analyzer()
analyzer.analyze(text="Échauffement du modèle.", language="fr")  # le 1er appel « à froid » n'est pas mesuré
m1.detect("Échauffement des règles.")                            # idem : charge et compile m1.yaml

print(f"Prompts : {len(prompts)} ({prompts.sensible.sum()} sensibles, {(prompts.sensible == 0).sum()} non sensibles "
      f"dont {prompts.piege.sum()} pièges)")
print(f"Règles M1 : {len(m1.load_rules())} règles, lues depuis {origine_m1}")
print(f"Presidio : entités supportées en français = {sorted(analyzer.get_supported_entities(language='fr'))}")

Prompts : 150 (85 sensibles, 65 non sensibles dont 55 pièges)
Règles M1 : 8 règles, lues depuis origin/cyber (commit 1761758 2026-11-12)
Presidio : entités supportées en français = ['AGE', 'CRYPTO', 'DATE_TIME', 'EMAIL', 'EMAIL_ADDRESS', 'IBAN_CODE', 'ID', 'IP_ADDRESS', 'LOCATION', 'MAC_ADDRESS', 'MEDICAL_LICENSE', 'NRP', 'ORGANIZATION', 'PERSON', 'PHONE_NUMBER', 'URL', 'UUID']


## 1. Les trois détecteurs comparés

| Détecteur | Contenu | Pourquoi le tester |
|---|---|---|
| **Presidio natif** | Presidio FR sans réglage : regex intégrées (IBAN, e-mail, IP, URL…) + NER spaCy (personnes, lieux, organisations, dates) | c'est la tâche ★ : mesurer l'outil « tel quel » |
| **Règles M1** | `detect()` d'Aloïs : 8 règles regex + validation (IBAN mod 97, Luhn, clé NIR, entropie…) | c'est ce qui est déjà prêt pour le proxy |
| **Hybride** | Règles M1 **+** les seules entités Presidio utiles : `PERSON`, `EMAIL_ADDRESS`, `PHONE_NUMBER`, `IBAN_CODE` | piste de « modèle » proposée : les règles pour les formats fixes, le NER pour les noms |

Pour l'hybride, on écarte volontairement `LOCATION`, `ORGANIZATION`, `URL`, `DATE_TIME`, `NRP`, `UUID` : en partie 3 on voit qu'ils créent surtout des fausses alertes (« Traduis », « Dockerfile », « TVA », « France »…).

Les trois renvoient une liste de détections `{type, debut, fin}`, comme `detect()` dans `docs/interface-detect.md`. **Un prompt déclenche une alerte s'il y a au moins une détection.**

In [2]:
ENTITES_HYBRIDE = {"PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "IBAN_CODE"}


def detect_presidio(texte):
    return [{"type": r.entity_type, "debut": r.start, "fin": r.end, "valeur": texte[r.start:r.end]}
            for r in analyzer.analyze(text=texte, language="fr")]


def detect_m1(texte):
    return [{"type": f["regle"], "debut": f["debut"], "fin": f["fin"], "valeur": f["extrait_masque"]}
            for f in m1.detect(texte)]


def detect_hybride(texte):
    return detect_m1(texte) + [d for d in detect_presidio(texte) if d["type"] in ENTITES_HYBRIDE]


DETECTEURS = {"Presidio natif": detect_presidio, "Règles M1": detect_m1, "Hybride": detect_hybride}

# On passe les 150 prompts dans chaque détecteur, en mesurant le temps de chaque appel.
lignes = []
for _, p in prompts.iterrows():
    for nom, detect in DETECTEURS.items():
        t0 = time.perf_counter()
        trouves = detect(p.texte)
        duree_ms = (time.perf_counter() - t0) * 1000
        lignes.append({"id": p.id, "detecteur": nom, "detections": trouves,
                       "types_trouves": sorted({d["type"] for d in trouves}),
                       "alerte": bool(trouves), "duree_ms": duree_ms})
resultats = pd.DataFrame(lignes).merge(prompts, on="id")
print(f"{len(resultats)} analyses ({len(prompts)} prompts x {len(DETECTEURS)} détecteurs)")

450 analyses (150 prompts x 3 détecteurs)


### Correspondance entre les types de Nolan et ce que chaque détecteur sait produire

Chaque prompt sensible de `prompts_v1.csv` liste ses données dans `types_donnees` (`NOM`, `IBAN`, `MOT_DE_PASSE`…).
On dit qu'une donnée est **trouvée** si le détecteur renvoie un type compatible. Si aucun type n'est compatible, le détecteur **ne peut pas** la trouver : c'est le travail d'un autre composant.

In [3]:
# type Nolan -> (types Presidio compatibles, règles M1 compatibles)
CORRESPONDANCE = {
    "NOM":            ({"PERSON"}, set()),
    "ADRESSE":        ({"LOCATION"}, set()),
    "DATE_NAISSANCE": ({"DATE_TIME"}, set()),
    "TEL":            ({"PHONE_NUMBER"}, {"telephone"}),
    "EMAIL":          ({"EMAIL_ADDRESS"}, {"email"}),
    "IBAN":           ({"IBAN_CODE"}, {"iban"}),
    "CARTE":          ({"CREDIT_CARD"}, {"carte_bancaire"}),   # CREDIT_CARD n'existe pas en FR dans Presidio
    "NIR":            (set(), {"nir"}),
    "SIRET":          (set(), {"siret"}),
    "CLE_API":        (set(), {"cle_api"}),
    "TOKEN":          (set(), {"cle_api"}),
    "SECRET":         (set(), {"cle_api", "mot_de_passe"}),
    "CLE_PRIVEE":     (set(), {"cle_api"}),
    "URL_SECRETE":    ({"URL"}, {"cle_api"}),
    "MOT_DE_PASSE":   (set(), {"mot_de_passe"}),
    "IP_INTERNE":     ({"IP_ADDRESS"}, set()),
}
# Données qui ne sont pas un « format » : seul un classifieur de prompt peut les voir.
CLASSIFIEUR = {"SANTE", "SALAIRE", "SOLDE", "DETTE", "PRET", "DONNEES_COMMERCIALES"}
# Données à format fixe sans règle M1 aujourd'hui.
REGLE_A_AJOUTER = {"PASSEPORT", "CNI", "PERMIS", "IMMATRICULATION", "LICENCE"}
# Données couvertes par une règle M1 qui en rate encore une bonne partie (voir plus bas).
REGLE_A_AMELIORER = {"MOT_DE_PASSE", "CLE_API", "SECRET", "URL_SECRETE"}

TYPES_CRITIQUES = {"IBAN", "CARTE", "NIR", "CLE_API", "TOKEN", "SECRET", "CLE_PRIVEE", "URL_SECRETE", "MOT_DE_PASSE"}


def compatibles(type_nolan, detecteur):
    presidio, regles = CORRESPONDANCE.get(type_nolan, (set(), set()))
    if detecteur == "Presidio natif":
        return presidio
    if detecteur == "Règles M1":
        return regles
    return regles | (presidio & ENTITES_HYBRIDE)


def trouve(ligne, type_nolan):
    return bool(compatibles(type_nolan, ligne.detecteur) & set(ligne.types_trouves))


resultats["types_trouves_ok"] = resultats.apply(lambda l: [t for t in l.types if trouve(l, t)], axis=1)
resultats["types_rates"] = resultats.apply(lambda l: [t for t in l.types if not trouve(l, t)], axis=1)

## 2. ★ Test sur 20 prompts : ce que Presidio trouve, ce qu'il rate

Les 20 prompts sont choisis à la main dans `prompts_v1.csv` pour couvrir **chaque type de donnée** et **chaque sorte de piège** du guide d'annotation :

- **12 sensibles** : identité (×2), IBAN, carte (×2), SIRET, liste d'e-mails clients, clé API, mot de passe en langage courant, chaîne de connexion à une base, santé, salaire.
- **8 non sensibles piégeux** : IBAN de documentation, carte de test Stripe, IBAN masqué, variable d'API vide, personnage public, prénom seul, empreinte SHA-256, nom de pays.

In [4]:
SELECTION_20 = [5, 46, 52, 128, 137, 15, 83, 60, 50, 142, 32, 21,      # sensibles
                148, 12, 65, 100, 145, 136, 86, 26]                     # non sensibles piégeux


def resume(detections):
    return ", ".join(f"{d['type']} «{d['valeur'][:25]}»" for d in detections) or "—"


def verdict(l):
    if l.sensible == 0:
        return "✅ rien signalé" if not l.alerte else "⚠️ fausse alerte"
    if not l.types_rates:
        return "✅ tout trouvé"
    return "🟠 en partie" if l.types_trouves_ok else "❌ raté"


vue = resultats[resultats.id.isin(SELECTION_20)].copy()
vue["ordre"] = vue.id.map({i: n for n, i in enumerate(SELECTION_20)})
vue["verdict"] = vue.apply(verdict, axis=1)

presidio_20 = vue[vue.detecteur == "Presidio natif"].sort_values("ordre")
tableau_20 = pd.DataFrame({
    "id": presidio_20.id,
    "catégorie": presidio_20.categorie,
    "prompt (début)": presidio_20.texte.str.slice(0, 70) + "…",
    "attendu": presidio_20.types_donnees.replace("", "—"),
    "Presidio signale": presidio_20.detections.apply(resume),
    "Presidio rate": presidio_20.types_rates.apply(lambda t: ", ".join(t) or "—"),
    "Presidio": presidio_20.verdict,
}).set_index("id")
tableau_20

,catégorie,prompt (début),attendu,Presidio signale,Presidio rate,Presidio
id,,,,,,
5,identite,"Résume ce dossier candidat en 3 lignes : Thomas Lefèvre, 34 ans, 8 rue…",NOM|ADRESSE|TEL|EMAIL,"EMAIL_ADDRESS «thomas.lefevre87@gmail.co», PERSON «Thomas Lefèvre», LOCATION «rue Past...",—,✅ tout trouvé
46,identite,"Prépare la DPAE pour le nouvel embauché : Julien Marchetti, né le 02/0…",NOM|DATE_NAISSANCE|NIR|ADRESSE,"PERSON «Julien Marchetti», LOCATION «NIR», LOCATION «rue des Lilas», LOCATION «Talence...",NIR,🟠 en partie
52,finance,Vérifie que ce RIB est correct avant que je lance le virement de 4 800…,IBAN,"IBAN_CODE «FR76300043801961489574055», LOCATION «FR76300043801961489574055»",—,✅ tout trouvé
128,finance,"Écris un mail à ma banque pour contester un débit de 249,99 € sur ma c…",CARTE,PHONE_NUMBER «4970 6110 5273 0654»,CARTE,❌ raté
137,finance,"Le client a payé avec la carte 5132 9693 1819 0457, exp 09/28, cryptog…",CARTE,DATE_TIME «09/28»,CARTE,❌ raté
15,donnees_clients,Fais un tableau des factures impayées : SAS Bati-Ouest (SIRET 224 965 …,SIRET|NOM,"PERSON «Coiffure Elsa», PERSON «Yves Caron»",SIRET,🟠 en partie
83,donnees_clients,Ma boucle d'envoi de newsletter plante sur cette liste : ['evadubois50…,EMAIL,"EMAIL_ADDRESS «evadubois50@hotmail.fr», EMAIL_ADDRESS «david.eva@sfr.fr», EMAIL_ADDRES...",—,✅ tout trouvé
60,secret_technique,"Mon appel à l'API OpenAI renvoie 401 : client = OpenAI(api_key=""sk-pro…",CLE_API,—,CLE_API,❌ raté
50,secret_technique,Rédige un mail au prestataire pour lui donner l'accès VPN : identifian…,MOT_DE_PASSE,"URL «n.vi», URL «vpn.atelier-durand.fr»",MOT_DE_PASSE,❌ raté


Même sélection, verdict des trois détecteurs côte à côte :

In [5]:
cote_a_cote = (vue.pivot_table(index=["ordre", "id", "categorie"], columns="detecteur", values="verdict", aggfunc="first")
                  [list(DETECTEURS)].reset_index(level="ordre", drop=True))
cote_a_cote

,detecteur,Presidio natif,Règles M1,Hybride
id,categorie,,,
5,identite,✅ tout trouvé,🟠 en partie,🟠 en partie
46,identite,🟠 en partie,🟠 en partie,🟠 en partie
52,finance,✅ tout trouvé,✅ tout trouvé,✅ tout trouvé
128,finance,❌ raté,✅ tout trouvé,✅ tout trouvé
137,finance,❌ raté,✅ tout trouvé,✅ tout trouvé
15,donnees_clients,🟠 en partie,🟠 en partie,✅ tout trouvé
83,donnees_clients,✅ tout trouvé,✅ tout trouvé,✅ tout trouvé
60,secret_technique,❌ raté,✅ tout trouvé,✅ tout trouvé
50,secret_technique,❌ raté,❌ raté,❌ raté


In [6]:
print("Sur les 20 prompts :")
for nom in DETECTEURS:
    v = vue[vue.detecteur == nom]
    sens, non_sens = v[v.sensible == 1], v[v.sensible == 0]
    print(f"  {nom:<15} sensibles signalés {sens.alerte.sum():>2}/12 · données entièrement trouvées "
          f"{(sens.types_rates.str.len() == 0).sum():>2}/12 · fausses alertes {non_sens.alerte.sum()}/8")

Sur les 20 prompts :
  Presidio natif  sensibles signalés 11/12 · données entièrement trouvées  3/12 · fausses alertes 7/8
  Règles M1       sensibles signalés  8/12 · données entièrement trouvées  5/12 · fausses alertes 2/8
  Hybride         sensibles signalés 10/12 · données entièrement trouvées  6/12 · fausses alertes 6/8


### Ce qu'on retient des 20 prompts

**Presidio trouve bien :**
- les **noms de personnes** (Thomas Lefèvre, Karim Aubert, Damien Roche…) grâce au NER spaCy ;
- les **e-mails**, les **téléphones français** et les **IBAN** (avec la vérification de la clé) ;
- les **dates de naissance** (format `jj/mm/aaaa`).

**Presidio rate :**
- les **cartes bancaires** (4970 6110 5273 0654, 5132 9693 1819 0457) : pas de détecteur de carte en français ;
- le **n° de sécurité sociale** (NIR) et le **SIRET** : pas de détecteur français ;
- **tous les secrets techniques** : clé `sk-proj-…`, mot de passe `Tigre33$`, mot de passe dans `postgresql://admin:…@` ;
- la **santé** (« en arrêt pour dépression ») et le **salaire** : ce n'est pas un format, Presidio ne le voit pas. Le prompt est signalé seulement parce qu'il contient un nom.

**Presidio lève des fausses alertes sur les pièges :**
- l'IBAN de documentation de la banque (148) : la clé est valide, donc ni Presidio ni les règles ne peuvent savoir que c'est un exemple ;
- « Victor Hugo », « Paul », « API_KEY » ou une empreinte SHA-256 pris pour des **personnes**, « France » pris pour un **lieu**.

**Les règles M1 d'Aloïs** font l'inverse : elles trouvent carte, NIR, SIRET et clé API, et ne se trompent presque pas sur les pièges. Mais elles ne voient **aucun nom** et ratent les mots de passe écrits sans `:` ni `=` (« mot de passe Tigre33$ »).

## 3. ★ Les métriques retenues et leurs objectifs

On mesure **au niveau du prompt**, parce que c'est ce que vit l'utilisateur : son prompt part, ou il est bloqué/masqué.

| Métrique | Définition | Objectif | Pourquoi ce seuil |
|---|---|---|---|
| **Rappel** *(prioritaire)* | part des prompts **sensibles** qui déclenchent une alerte | **≥ 90 %** | une fuite ratée est irréversible : c'est le chiffre qui compte le plus |
| **Rappel critique** | part des prompts avec IBAN, carte, NIR, clé/token, mot de passe où **cette donnée précise** est trouvée | **≥ 95 %** | ce sont les données qu'on bloque ; on doit trouver la donnée elle-même pour la masquer |
| **Taux de fausses alertes** | part des prompts **non sensibles** qui déclenchent une alerte | **≤ 10 %** | au-delà, les salariés contournent l'outil ; ≈ 1 prompt légitime sur 10 dérangé au maximum |
| **Taux de fausses alertes sur pièges** | idem, sur les seuls prompts piégeux | **≤ 20 %** | les pièges sont faits pour tromper : on tolère plus |
| **Temps de réponse** | durée de `detect()` par prompt, 95ᵉ centile (p95) | **≤ 50 ms** | le proxy ajoute ce délai à chaque requête ; 50 ms reste imperceptible |
| Précision par donnée *(suivi)* | part des détections qui tombent sur une vraie donnée | ≥ 85 % | mesurable seulement sur AI4Privacy (positions annotées) |

**Attention : taux de fausses alertes ≠ 100 % − précision.** La précision de 90 % mesurée sur AI4Privacy se calcule *par donnée*, sur des textes qui contiennent tous des données. Le taux de fausses alertes se calcule *par prompt*, sur des prompts qui n'en contiennent pas. C'est pour ça que les pièges de Nolan sont indispensables.

In [7]:
OBJECTIFS = {"Rappel": (0.90, "≥"), "Rappel critique": (0.95, "≥"), "Fausses alertes": (0.10, "≤"),
             "Fausses alertes (pièges)": (0.20, "≤"), "Temps p95 (ms)": (50, "≤")}


def p95(valeurs):
    v = sorted(valeurs)
    return v[int(0.95 * (len(v) - 1))]


def metriques(r):
    sens, non_sens = r[r.sensible == 1], r[r.sensible == 0]
    critiques = [(t, l) for _, l in sens.iterrows() for t in l.types if t in TYPES_CRITIQUES]
    return {
        "Rappel": sens.alerte.mean(),
        "Rappel critique": sum(trouve(l, t) for t, l in critiques) / len(critiques),
        "Fausses alertes": non_sens.alerte.mean(),
        "Fausses alertes (pièges)": non_sens[non_sens.piege == 1].alerte.mean(),
        "Temps médian (ms)": statistics.median(r.duree_ms),
        "Temps p95 (ms)": p95(r.duree_ms),
    }


tableau_metriques = pd.DataFrame({nom: metriques(resultats[resultats.detecteur == nom]) for nom in DETECTEURS}).T


def atteint(valeur, metrique):
    if metrique not in OBJECTIFS:
        return ""
    seuil, sens = OBJECTIFS[metrique]
    return "✅" if (valeur >= seuil if sens == "≥" else valeur <= seuil) else "❌"


affichage = tableau_metriques.copy().astype(object)
for col in tableau_metriques.columns:
    for det in tableau_metriques.index:
        v = tableau_metriques.loc[det, col]
        texte = f"{v:.1f}" if "ms" in col else f"{v:.0%}"
        affichage.loc[det, col] = f"{texte} {atteint(v, col)}".strip()
affichage.loc["Objectif"] = [f"{OBJECTIFS[c][1]} {OBJECTIFS[c][0] if 'ms' in c else format(OBJECTIFS[c][0], '.0%')}"
                             if c in OBJECTIFS else "" for c in affichage.columns]
print(f"Sur les 150 prompts de data/prompts_v1.csv "
      f"({prompts.sensible.sum()} sensibles, {(prompts.sensible == 0).sum()} non sensibles dont {prompts.piege.sum()} pièges)")
affichage

Sur les 150 prompts de data/prompts_v1.csv (85 sensibles, 65 non sensibles dont 55 pièges)


,Rappel,Rappel critique,Fausses alertes,Fausses alertes (pièges),Temps médian (ms),Temps p95 (ms)
Presidio natif,94% ✅,30% ❌,40% ❌,45% ❌,8.0,11.5 ✅
Règles M1,60% ❌,72% ❌,8% ✅,9% ✅,0.1,0.1 ✅
Hybride,88% ❌,72% ❌,17% ❌,20% ✅,5.4,8.3 ✅
Objectif,≥ 90%,≥ 95%,≤ 10%,≤ 20%,,≤ 50


**Lecture :**
- **Presidio natif** a un bon rappel (94 %), mais **40 % des prompts non sensibles** déclenchent une alerte : inutilisable tel quel. Son rappel est en partie « chanceux » : un prompt de santé est signalé parce qu'il contient un nom, pas parce que Presidio a compris qu'il parle de santé. Et il ne trouve que 30 % des données critiques (ni carte, ni NIR, ni secret).
- **Les règles M1** sont précises (8 % de fausses alertes) et très rapides (< 1 ms), mais elles ratent tous les prompts qui ne contiennent qu'un nom, une adresse ou une info de santé (rappel 60 %).
- **L'hybride** réunit le meilleur des deux : rappel 88 %, fausses alertes divisées par plus de deux par rapport à Presidio (17 %). Il n'atteint pas encore les objectifs : rappel critique 72 % (mots de passe et certains secrets ratés) et 17 % de fausses alertes. La partie 4 montre comment combler l'écart.
- **Le temps de réponse** est largement dans l'objectif pour les trois.

## 4. Règles d'Aloïs ou modèle ? Rappel par type de donnée

Pour chaque type de donnée présent dans les prompts sensibles : part des cas où **la donnée elle-même** est trouvée.
La dernière colonne dit quel composant doit s'en charger.

In [8]:
def responsable(t):
    if t in CLASSIFIEUR:
        return "classifieur (pas un format)"
    if t in REGLE_A_AJOUTER:
        return "règle à ajouter"
    if t in REGLE_A_AMELIORER:
        return "règles M1 (à améliorer)"
    presidio, regles = CORRESPONDANCE[t]
    if regles and t not in {"TEL", "EMAIL"}:
        return "règles M1"
    if t in {"TEL", "EMAIL"}:
        return "règles M1 (Presidio aussi bon)"
    if t == "NOM":
        return "modèle NER (Presidio/spaCy)"
    if t == "ADRESSE":
        return "modèle NER (LOCATION, à filtrer)"
    return "règle à ajouter (Presidio la voit)"


sens = resultats[resultats.sensible == 1]
lignes = []
for t, n in Counter(t for ts in prompts[prompts.sensible == 1].types for t in ts).most_common():
    ligne = {"type": t, "nb prompts": n}
    for nom in DETECTEURS:
        sous = sens[(sens.detecteur == nom) & sens.types.apply(lambda ts: t in ts)]
        ligne[nom] = f"{sous.apply(lambda l: trouve(l, t), axis=1).mean():.0%}"
    ligne["qui doit s'en charger"] = responsable(t)
    lignes.append(ligne)
par_type = pd.DataFrame(lignes).set_index("type")
par_type

,nb prompts,Presidio natif,Règles M1,Hybride,qui doit s'en charger
type,,,,,
NOM,51,98%,0%,98%,modèle NER (Presidio/spaCy)
TEL,16,100%,100%,100%,règles M1 (Presidio aussi bon)
SANTE,12,0%,0%,0%,classifieur (pas un format)
EMAIL,11,100%,100%,100%,règles M1 (Presidio aussi bon)
IBAN,11,100%,100%,100%,règles M1
DATE_NAISSANCE,10,100%,0%,0%,règle à ajouter (Presidio la voit)
ADRESSE,10,100%,0%,0%,"modèle NER (LOCATION, à filtrer)"
MOT_DE_PASSE,8,0%,12%,12%,règles M1 (à améliorer)
NIR,7,0%,100%,100%,règles M1


Et par catégorie du guide d'annotation (rappel au niveau du prompt) :

In [9]:
par_categorie = (resultats[resultats.sensible == 1]
                 .pivot_table(index="categorie", columns="detecteur", values="alerte", aggfunc="mean")
                 [list(DETECTEURS)].map(lambda v: f"{v:.0%}"))
par_categorie["nb prompts"] = prompts[prompts.sensible == 1].categorie.value_counts()
par_categorie

detecteur,Presidio natif,Règles M1,Hybride,nb prompts
categorie,,,,
donnees_clients,100%,89%,100%,19
finance,100%,83%,100%,18
identite,100%,67%,100%,18
sante,100%,8%,100%,12
secret_technique,72%,33%,44%,18


### Pourquoi les règles ratent certains secrets

Les secrets ratés par les règles M1 (prompts sensibles sans aucune détection de `cle_api` ni `mot_de_passe`) :

In [10]:
secrets = resultats[(resultats.detecteur == "Règles M1") & (resultats.categorie == "secret_technique")]
rates = secrets[secrets.types_rates.str.len() > 0]
rates[["id", "types_donnees", "texte"]].assign(texte=rates.texte.str.slice(0, 110)).set_index("id")

,types_donnees,texte
id,,
7,LICENCE,"Le logiciel de compta demande la clé de licence, la voici : J7BZW-ZEQYM-58Q8W-B74GX-RK..."
23,CLE_API,"Ajoute un retry à cette requête : requests.post(url, headers={'Authorization': 'Bearer..."
50,MOT_DE_PASSE,"Rédige un mail au prestataire pour lui donner l'accès VPN : identifiant n.vidal, mot d..."
54,SECRET,"Traduis ce Terraform en Ansible : provider ""azurerm"" { client_id = ""e2b770ee-1241-450c..."
58,MOT_DE_PASSE,Rédige un mail au DSI : le compte root du serveur web a pour mot de passe SUx%fVbri9&P...
67,URL_SECRETE,"Mon webhook Slack ne poste rien, voilà l'URL : https://hooks.slack.com/services/T0P4Z8..."
112,CLE_API,Corrige mon fichier .env : STRIPE_SECRET_KEY=sk_live_FAKE_PWlbSHI3WobHz7iMflc1ZYvG SEN...
118,MOT_DE_PASSE,Résume ce ticket : l'utilisateur ne sait plus changer le mot de passe admin du NAS (ac...
120,MOT_DE_PASSE|IP_INTERNE,Mon script de sauvegarde FTP plante : ftp = FTP('192.168.10.20'); ftp.login('backup_ad...


Trois causes, à remonter à Aloïs :

1. **Mot de passe sans séparateur.** La règle `mot_de_passe` attend `:`, `=` ou « est » après le mot-clé. Elle rate « mot de passe Tigre33$ », « mot de passe provisoire Atlantique123! », « (actuel : Printemps64) », ainsi que les mots de passe dans du code (`ftp.login('backup_admin', 'Garonne2024$')`, `postgresql://admin:Marine2026@@10.2.4.15`).
2. **Formats de secrets absents :** jeton `Bearer …` opaque, URL de webhook Slack, `client_secret` Azure, clé SendGrid `SG.…`, clé de licence logicielle.
3. **Artefact du jeu de prompts :** la clé `sk_live_FAKE_…` (prompt 112) contient `FAKE_` pour ne pas être bloquée par GitHub. La règle Stripe exige des caractères alphanumériques juste après `sk_live_`. Ce n'est pas un vrai défaut de la règle.

Une piste simple, sans modèle : une règle « **chaîne à forte entropie** près d'un mot-clé » (`secret`, `token`, `key`, `password`, `Bearer`), avec la validation `entropie` qui existe déjà dans `m1_detect.py`.

### Les fausses alertes restantes de l'hybride

In [11]:
fa = resultats[(resultats.detecteur == "Hybride") & (resultats.sensible == 0) & resultats.alerte]
fa[["id", "texte", "detections"]].assign(
    texte=fa.texte.str.slice(0, 80), detections=fa.detections.apply(resume)).set_index("id")

,texte,detections
id,,
12,"Mes tests Stripe échouent avec la carte de test 4242 4242 4242 4242, exp 12/34,",carte_bancaire «**** **** **** 4242»
28,"Traduis en anglais : « Pour tout problème de paiement, écrivez à support@example","email «s******************», EMAIL_ADDRESS «support@example.com»"
38,C'est quoi les 3 parties de ce JWT d'exemple de jwt.io ? eyJhbGciOiJIUzI1NiIsInR,cle_api «eyJhbG*******************»
49,"Écris une histoire courte pour enfants avec Léa, Tom et leur chien Biscuit.",PERSON «Léa»
86,À quoi correspond ce SHA-256 : e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca,PERSON «e3b0c44298fc1c149afbf4c89»
100,"Dans mon .env.example j'ai mis API_KEY=your_api_key_here, c'est la bonne pratiqu",PERSON «API_KEY»
103,Écris un mail à Mme Martin du service achats pour lui demander le bon de command,PERSON «Mme Martin»
106,"Comment hasher un mot de passe avec bcrypt en Python ? J'ai : password = input(""","mot_de_passe «******», PERSON «hasher»"
136,Écris un mail à Paul pour décaler la réunion de lundi à 15 h.,PERSON «Paul»


Deux familles :
- **Exemples publics à clé valide** (carte de test Stripe `4242…`, JWT de jwt.io, IBAN de documentation, `support@example.com`) : on peut les mettre dans une **liste blanche** de valeurs de test connues.
- **Prénoms et personnages publics** (« Paul », « Mme Martin », « Victor Hugo ») et mots pris pour des noms (« hasher », « API_KEY », une empreinte SHA-256) : un nom seul ne suffit pas à rendre un prompt sensible. C'est le rôle du **classifieur** de prompt : décider si le nom est associé à une donnée personnelle.

## 5. Rappel : Presidio sur le dataset public AI4Privacy

Résultats détaillés dans `docs/analyse-dataset-ai4privacy.md` (1 000 textes français, graine 42).
Recalcul : `python data/scripts/preparer_dataset.py` puis `python data/scripts/evaluer_presidio.py`.

In [12]:
ai4p = pd.read_csv(RACINE / "data" / "resultats" / "ai4privacy_par_label.csv", encoding="utf-8-sig")
ai4p.set_index("label_ai4privacy")

,type_presidio_attendu,nb,rappel_type_%,rappel_tous_types_%
label_ai4privacy,,,,
DATE,DATE_TIME,865,33.4,34.3
GIVENNAME,PERSON,837,68.6,77.5
SURNAME,PERSON,738,61.7,67.5
EMAIL,EMAIL_ADDRESS,515,100.0,100.0
CITY,LOCATION,497,73.6,86.1
TITLE,-,467,NaN,62.7
TELEPHONENUM,PHONE_NUMBER,387,50.4,52.5
AGE,AGE,379,0.0,0.0
ZIPCODE,-,337,NaN,21.4


Ces chiffres confirment le test sur prompts : Presidio est bon sur **e-mails, noms et lieux**, nul sur **cartes bancaires** et **âges** en français.
Mais AI4Privacy ne contient **ni IBAN, ni clés API, ni mots de passe**, et ses textes ne ressemblent pas à des prompts. Le jeu de Nolan reste donc la référence pour les métriques de CyberSentinel.

## 6. Sauvegarde des résultats

In [13]:
dossier = RACINE / "data" / "resultats"
dossier.mkdir(exist_ok=True)

# Détail par prompt et par détecteur : seulement les types trouvés, jamais la valeur (RGPD).
(resultats[["id", "detecteur", "sensible", "categorie", "piege", "types_donnees", "alerte",
            "types_trouves", "types_rates", "duree_ms"]]
 .assign(types_trouves=lambda d: d.types_trouves.str.join("|"),
         types_rates=lambda d: d.types_rates.str.join("|"),
         duree_ms=lambda d: d.duree_ms.round(2))
 .to_csv(dossier / "prompts_detail.csv", index=False, encoding="utf-8-sig"))
tableau_metriques.round(4).to_csv(dossier / "prompts_metriques.csv", encoding="utf-8-sig")
par_type.to_csv(dossier / "prompts_rappel_par_type.csv", encoding="utf-8-sig")
print("Enregistré :", *sorted(p.name for p in dossier.glob("prompts_*.csv")), sep="\n  ")

Enregistré :
  prompts_detail.csv
  prompts_metriques.csv
  prompts_rappel_par_type.csv


## 7. Conclusion

| Question de la fiche | Réponse |
|---|---|
| Ce que Presidio trouve | noms, e-mails, téléphones FR, IBAN, dates |
| Ce que Presidio rate | cartes bancaires, NIR, SIRET, tous les secrets techniques, la santé ; et trop de fausses alertes (organisations, lieux, prénoms isolés) |
| Ce que les règles suffisent à attraper | IBAN, carte, NIR, SIRET, téléphone, e-mail, clés API à préfixe connu, JWT, clés privées |
| Ce qui demande un modèle | noms de personnes et adresses (NER) ; santé, salaire, données commerciales, et « ce nom est-il lié à une donnée personnelle ? » (classifieur de prompt) |
| Métriques retenues | rappel ≥ 90 % (critique ≥ 95 %), fausses alertes ≤ 10 % (pièges ≤ 20 %), p95 ≤ 50 ms |

**Choix proposé :** un `detect()` en trois étages, détaillé dans `docs/choix-modele-metriques.md` :
1. **règles M1** (Aloïs), améliorées pour les secrets ;
2. **Presidio/spaCy**, limité à `PERSON` (+ e-mail, téléphone, IBAN en double sécurité) ;
3. **classifieur de prompt** sur les 6 étiquettes d'Elias (baseline TF-IDF de Nolan, puis CamemBERT si besoin), qui décide pour la santé, les finances non formatées, et filtre les noms isolés.